# Tratamento de Dados - ENEM 2023

Carrega os microdados brutos, seleciona e tipa as colunas de interesse, aplica os filtros de qualidade e exporta a base tratada em Parquet para o notebook de análise.

In [1]:
import pandas as pd

DADOS_PATH = r'..\data\raw\microdados_enem_2023\DADOS\MICRODADOS_ENEM_2023.csv'

## 1. Dicionário de variáveis

Mapeamento construído manualmente a partir do dicionário oficial dos Microdados
ENEM 2023, disponível em `data/raw/microdados_enem_2023/DICIONÁRIO/`.

Para este trabalho, foi realizada uma seleção das variáveis consideradas mais relevantes para a análise. Das 76 variáveis originais disponíveis na base de dados, foram selecionadas 34 para o carregamento. Após os filtros, a coluna de treineiro é descartada (fica constante), e a base final exportada tem 33 colunas.

In [2]:
# Coluna original -> Nome legível (possível de ser identificado sem consultar o dicionário)
rename_map = {
    # Dados do Participante (9 itens)
    'NU_INSCRICAO':          'Numero_Inscricao',
    'NU_ANO':                'Ano',
    'TP_FAIXA_ETARIA':       'Faixa_Etaria',
    'TP_SEXO':               'Sexo',
    'TP_COR_RACA':           'Cor_Raca',
    'TP_ST_CONCLUSAO':       'Situacao_Conclusao_EM',
    'TP_ANO_CONCLUIU':       'Ano_Conclusao_EM',
    'TP_ESCOLA':             'Tipo_Escola',
    'TP_ENSINO':             'Tipo_Ensino',
    # Dados da Escola (4 itens)
    'NO_MUNICIPIO_ESC':      'Nome_Municipio_Escola',
    'SG_UF_ESC':             'UF_Escola',
    'TP_DEPENDENCIA_ADM_ESC':'Dependencia_Adm_Escola',
    'TP_LOCALIZACAO_ESC':    'Localizacao_Escola',
    # Dados das Provas Objetivas (8 itens)
    'TP_PRESENCA_CN':        'Presenca_Ciencias_Natureza',
    'TP_PRESENCA_CH':        'Presenca_Ciencias_Humanas',
    'TP_PRESENCA_LC':        'Presenca_Linguagens',
    'TP_PRESENCA_MT':        'Presenca_Matematica',
    'NU_NOTA_CN':            'Nota_Ciencias_Natureza',
    'NU_NOTA_CH':            'Nota_Ciencias_Humanas',
    'NU_NOTA_LC':            'Nota_Linguagens',
    'NU_NOTA_MT':            'Nota_Matematica',
    # Dados da Redação (2 itens)
    'TP_STATUS_REDACAO':     'Status_Redacao',
    'NU_NOTA_REDACAO':       'Nota_Redacao',
    # Questionário Socioeconômico (10 itens)
    'Q001': 'Q1_Esc_Pai',
    'Q002': 'Q2_Esc_Mae',
    'Q003': 'Q3_Ocup_Pai',
    'Q004': 'Q4_Ocup_Mae',
    'Q005': 'Q5_Pessoas_Resid',
    'Q006': 'Q6_Renda_Fam',
    'Q008': 'Q8_Banheiro',
    'Q012': 'Q12_Geladeira',
    'Q024': 'Q24_Computador',
    'Q025': 'Q25_Internet',
}

## 2. Carregamento dos dados

In [3]:
dtype_map = {
    'NU_INSCRICAO': str,
    'NU_ANO': 'int16',
    'TP_FAIXA_ETARIA': 'Int8',
    'TP_SEXO': str,
    'TP_COR_RACA': 'Int8',
    'TP_ST_CONCLUSAO': 'Int8',
    'TP_ANO_CONCLUIU': 'Int8',
    'TP_ESCOLA': 'Int8',
    'TP_ENSINO': 'Int8',
    'IN_TREINEIRO': 'Int8',
    'NO_MUNICIPIO_ESC': str,
    'SG_UF_ESC': str,
    'TP_DEPENDENCIA_ADM_ESC': 'Int8',
    'TP_LOCALIZACAO_ESC': 'Int8',
    'TP_PRESENCA_CN': 'Int8',
    'TP_PRESENCA_CH': 'Int8',
    'TP_PRESENCA_LC': 'Int8',
    'TP_PRESENCA_MT': 'Int8',
    'NU_NOTA_CN': 'float32',
    'NU_NOTA_CH': 'float32',
    'NU_NOTA_LC': 'float32',
    'NU_NOTA_MT': 'float32',
    'TP_STATUS_REDACAO': 'Int8',
    'NU_NOTA_REDACAO': 'float32',
    'Q001': str,
    'Q002': str,
    'Q003': str,
    'Q004': str,
    'Q005': str,
    'Q006': str,
    'Q008': str,
    'Q012': str,
    'Q024': str,
    'Q025': str,
}

df = pd.read_csv(
    DADOS_PATH,
    sep=';',
    encoding='latin-1',
    usecols=list(dtype_map.keys()),
    dtype=dtype_map,
)
print(f'Shape original: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')

Shape original: (3933955, 34)
Colunas: ['NU_INSCRICAO', 'NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_COR_RACA', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU', 'TP_ESCOLA', 'TP_ENSINO', 'IN_TREINEIRO', 'NO_MUNICIPIO_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_REDACAO', 'Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006', 'Q008', 'Q012', 'Q024', 'Q025']


## 3. Filtros obrigatórios

Antes das etapas de análise, foram aplicados três filtros para garantir a consistência e a qualidade do conjunto de dados:
- Seleção apenas dos candidatos que compareceram a todas as provas objetivas do ENEM;
- Remoção dos treineiros (participantes que fazem a prova apenas para autoavaliação, sem valer para ingresso no ensino superior);
- Remoção de registros duplicados, utilizando o número de inscrição como identificador único.

In [4]:
# Filtro 1: candidatos presentes nas 4 provas objetivas
print(f'Registros antes do filtro de presença: {df.shape}')
df = df[
    (df['TP_PRESENCA_CN'] == 1) &
    (df['TP_PRESENCA_CH'] == 1) &
    (df['TP_PRESENCA_LC'] == 1) &
    (df['TP_PRESENCA_MT'] == 1)
]
print(f'Registros depois do filtro de presença: {df.shape}')

Registros antes do filtro de presença: (3933955, 34)
Registros depois do filtro de presença: (2678264, 34)


In [5]:
# Filtro 2: remoção de treineiros
print(f'Registros antes do filtro de treineiros: {df.shape}')
df = df[df['IN_TREINEIRO'] == 0]
df = df.drop(columns='IN_TREINEIRO')  # após o filtro a coluna fica constante
print(f'Registros depois do filtro de treineiros: {df.shape}')

Registros antes do filtro de treineiros: (2678264, 34)
Registros depois do filtro de treineiros: (2166843, 33)


In [6]:
# Filtro 3: remoção de duplicatas pelo número de inscrição
print(f'Registros antes da remoção de duplicatas: {df.shape}')
df = df.drop_duplicates(subset='NU_INSCRICAO')
print(f'Registros depois da remoção de duplicatas: {df.shape}')

Registros antes da remoção de duplicatas: (2166843, 33)
Registros depois da remoção de duplicatas: (2166843, 33)


## 4. Renomeação das colunas

In [7]:
df = df.rename(columns=rename_map)
print(f'Novas colunas: {df.columns.tolist()}')

Novas colunas: ['Numero_Inscricao', 'Ano', 'Faixa_Etaria', 'Sexo', 'Cor_Raca', 'Situacao_Conclusao_EM', 'Ano_Conclusao_EM', 'Tipo_Escola', 'Tipo_Ensino', 'Nome_Municipio_Escola', 'UF_Escola', 'Dependencia_Adm_Escola', 'Localizacao_Escola', 'Presenca_Ciencias_Natureza', 'Presenca_Ciencias_Humanas', 'Presenca_Linguagens', 'Presenca_Matematica', 'Nota_Ciencias_Natureza', 'Nota_Ciencias_Humanas', 'Nota_Linguagens', 'Nota_Matematica', 'Status_Redacao', 'Nota_Redacao', 'Q1_Esc_Pai', 'Q2_Esc_Mae', 'Q3_Ocup_Pai', 'Q4_Ocup_Mae', 'Q5_Pessoas_Resid', 'Q6_Renda_Fam', 'Q8_Banheiro', 'Q12_Geladeira', 'Q24_Computador', 'Q25_Internet']


## 5. Exportação dos dados tratados

In [8]:
PROCESSED_PATH = r'..\data\processed\enem_2023_tratado.parquet'
df.to_parquet(PROCESSED_PATH, index=False)
print(f'Dados exportados para {PROCESSED_PATH}')
print(f'Shape: {df.shape}')

Dados exportados para ..\data\processed\enem_2023_tratado.parquet
Shape: (2166843, 33)
